In [29]:
import chromadb
from sentence_transformers import SentenceTransformer

In [30]:
import json


In [31]:

from tqdm import tqdm
import openai

In [32]:
import pandas as pd

In [33]:
import numpy as np

In [34]:
df = pd.read_csv("DOB_NOW_cleaned.csv")

C:\Users\USER\AppData\Local\Temp\ipykernel_9668\1323950220.py:1: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("DOB_NOW_cleaned.csv")


In [35]:
# =========================================================
# 2. LOAD SEMANTIC CHUNKS
# =========================================================

with open(
    "embedding_documents.json",
    "r"
) as f:

    documents = json.load(f)

In [36]:
model = SentenceTransformer("all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [37]:
client = chromadb.PersistentClient(
    path="./chroma_db"
)

 

In [38]:
collection = client.get_or_create_collection(
    name="construction_risk_rag"
)

In [39]:
# documents = [

#     "Structural alteration project in Manhattan",

#     "Excavation and foundation work in Brooklyn",

#     "High-rise residential construction"
# ]

In [45]:
batch_size = 128

for i in tqdm(
    range(0, len(documents), batch_size)
):
    batch_docs = documents[i:i+batch_size]
    embeddings = model.encode(
        batch_docs
    ).tolist()

    ids = [

        f"project_{i+j}"

        for j in range(len(batch_docs))
    ]

    metadatas = []

    for j in range(len(batch_docs)):

        row = df.iloc[i + j]

        metadata = {

            "project_id": f"project_{i+j}",

            "borough": str(
                row.get("borough", "UNKNOWN")
            ),

            "job_type": str(
                row.get("job_type", "UNKNOWN")
            ),

            "bin": str(
                row.get("bin", "UNKNOWN")
            )
        }

        metadatas.append(metadata)

    collection.add(

        ids=ids,

        documents=batch_docs,

        embeddings=embeddings,

        metadatas=metadatas
    )
        
print("\nALL PROJECTS STORED IN CHROMADB!")
    


100%|██████████| 441/441 [37:47<00:00,  5.14s/it]


ALL PROJECTS STORED IN CHROMADB!


In [48]:
# =========================================================
# 8. TEST QUERY
# =========================================================

query = """

Foundation excavation project
with structural work in Manhattan

"""

# ---------------------------------------------------------
# CREATE QUERY EMBEDDING
# ---------------------------------------------------------

query_embedding = model.encode(
    [query]
).tolist()[0]

# ---------------------------------------------------------
# SEARCH CHROMADB
# ---------------------------------------------------------

results = collection.query(

    query_embeddings=[query_embedding],

    n_results=5
)

# =========================================================
# 9. DISPLAY RESULTS
# =========================================================

for i in range(len(results["documents"][0])):

    print(f"\n================ RESULT {i+1} ================\n")

    print("DOCUMENT:\n")

    print(results["documents"][0][i])

    print("\nMETADATA:\n")

    print(results["metadatas"][0][i])

    print("\n================================================")


================ RESULT 1 ================

DOCUMENT:



    PROJECT DETAILS:

    Job Type: Alteration
    Borough: MANHATTAN
    BIN: 1058275
    Block: 1920
    Lot: 26

    Building Type: Other

    Initial Cost: 70000.0

    Floor Area:
    700.0

    Existing Stories:
    3.0

    Proposed Stories:
    3.0

    Existing Dwelling Units:
    0.0

    Proposed Dwelling Units:
    0.0

    Filing Date:
    2026-01-02

    CONTRACTOR & SCOPE:

    Applicant Professional Title:
    PE

    Owner Business:
    NYC DDC

    Filing Representative:
    CITY WIDE EXPEDITING INC.

    Building Code:
    2022

    Structural Work:
    1

    Foundation Work:
    0

    Mechanical Systems Work:
    0

    General Construction Work:
    0

    Support Of Excavation:
    No

    Special Inspection Requirement:
    Post-installed anchors in masonry,Structural Steel - Details,Structural Steel - High Strength Bolting

    Progress Inspection Requirement:
    Final

    Scaffold:
    0

    Shed:
 